In [1]:
import torch
import numpy as np

In [2]:
data = [[1,2],[3,4]]
x_data = torch.tensor(data) # copy these data into tensor

In [3]:
np_array = np.array(data) 
x_np = torch.from_numpy(np_array)

In [4]:
x_ones = torch.ones_like(x_data) # retain the properties of x_data
print(f"Ones Tensor : \n {x_ones} \n")

Ones Tensor : 
 tensor([[1, 1],
        [1, 1]]) 



torch.ones_like(x_data)：做一个和 x_data 同形状、同数据类型、同设备的新张量，但每个元素都是 1


| 函数 | 填进去的数 |
|------|------------|
| `torch.ones_like(x)` | 全 1 |
| `torch.zeros_like(x)` | 全 0 |
| `torch.rand_like(x)` | 随机（通常要浮点张量） |
| `torch.empty_like(x)` | 未初始化的垃圾值 |



In [5]:
x_rand = torch.rand_like(x_data,dtype=torch.float) # random data
print(f"Random Tensor : \n {x_rand} \n")

Random Tensor : 
 tensor([[0.6059, 0.3085],
        [0.7840, 0.3904]]) 



In [6]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.9152, 0.8461, 0.2939],
        [0.2981, 0.0962, 0.9666]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


In [7]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


In [8]:
# We move our tensor to the GPU if avaliable

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "CPU"
tensor = tensor.to(device)
print(f"Tensor is stored on: {tensor.device}")

Tensor is stored on: mps:0




```python
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "CPU"
```

这是一行压缩的 if-else，等价于：

```python
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator().type
else:
    device = "CPU"
```

- `is_available()`：有没有加速器（Mac GPU / NVIDIA 等）？  
  - 你的电脑：**有** → 走进 `if`
- `current_accelerator().type`：加速器叫什么。你这里是字符串 `'mps'`。

所以跑完后：`device` 就是 `'mps'`。  
没有加速器的电脑：`device` 就是 `"CPU"`。

`device` 只是一个**字符串标签**（或设备对象），还没有搬数据。

---

```python
tensor = tensor.to(device)
```

- 右边：把当前的 `tensor`（当时在 CPU）**复制**到 `device` 指定的地方（你的是 MPS）。
- 左边：用同一个名字 `tensor` 接住这份新的。

之后 `tensor` 在 GPU 上；再打印 `.device` 就会是 `mps:0`。

---

## 串起来

```text
有加速器吗？──是──► device = "mps" ──► tensor.to("mps") ──► 数据在苹果 GPU
          └──否──► device = "CPU" ──► tensor.to("CPU") ──► 仍在 CPU
```


In [9]:
tensor = torch.ones(4, 4)
tensor[:,1] = 0
print(f"{tensor}\n")

tensor = torch.ones(4, 4)
tensor[:,1:] = 0
print(f"{tensor}\n")

tensor = torch.ones(4, 4)
tensor[:,1:3] = 0
print(f"{tensor}\n")



tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.]])

tensor([[1., 0., 0., 1.],
        [1., 0., 0., 1.],
        [1., 0., 0., 1.],
        [1., 0., 0., 1.]])



In [10]:
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(f"{t1}\n")

tensor([[1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 1.],
        [1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 1.],
        [1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 1.],
        [1., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 1.]])



dim=0 → 行方向（往下叠） 

dim=1 → 列方向（往右拼）

In [12]:
# this computes the element-wise product
print(f"tensor.mul(tensor) \n {tensor.mul(tensor)}\n")


tensor.mul(tensor) 
 tensor([[1., 0., 0., 1.],
        [1., 0., 0., 1.],
        [1., 0., 0., 1.],
        [1., 0., 0., 1.]])



这两行是：**对应位置相乘**（逐元素积），不是线性代数里的矩阵乘法。

```python
print(f"tensor.mul(tensor) \n {tensor.mul(tensor)}\n")
```

---

## `tensor.mul(tensor)` 在算什么

形状相同的两个张量（这里是自己乘自己），**同一个格子里的两个数相乘**，结果形状不变。

若 `tensor` 仍是之前那种 4×4（第二列为 0）：

```
1  0  1  1
1  0  1  1
1  0  1  1
1  0  1  1
```

则 `mul` 之后每个位置是 \(x_{ij} \times x_{ij} = x_{ij}^2\)：

```
1  0  1  1
1  0  1  1
1  0  1  1
1  0  1  1
```

（这里刚好还是 0 和 1，平方看不出变化；若格子是 `3`，结果会是 `9`。）

注释 *element-wise product* 就是这个意思。

---

## 等价写法

```python
tensor.mul(tensor)
tensor * tensor          # 最常见
torch.mul(tensor, tensor)
```

---

## 不要和矩阵乘法搞混

| | 逐元素 `mul` / `*` | 矩阵乘 `matmul` / `@` |
|--|-------------------|----------------------|
| 规则 | 同位相乘 | 行乘列再求和 |
| 形状 | 一般保持不变 | `4×4 @ 4×4` 仍是 4×4，但是另一种算法 |
| 例子 | `[[1,2],[3,4]] * 自己` → `[[1,4],[9,16]]` | `[[1,2],[3,4]] @ 自己` → `[[7,10],[15,22]]` |

教程下一格通常是 `tensor.matmul(tensor)` 或 `tensor @ tensor`，那才是矩阵乘。

---

## `print(f"... \n {tensor.mul(tensor)}\n")`

只是把计算结果嵌进字符串里打印。`\n` 是换行，方便看起来分块。

**一句话：** `mul` = 每个格子自己乘自己（或乘另一个同形状张量的对应格子）。

In [15]:
t1 = torch.rand(3,3)
t2 = torch.rand(3,3)

print(f"{t1}\n")
print(f"{t2}\n")

print(f"{torch.mul(t1,t2)}\n")

tensor([[0.2959, 0.2006, 0.4012],
        [0.5819, 0.4481, 0.0271],
        [0.9670, 0.2622, 0.8162]])

tensor([[0.5198, 0.3174, 0.8648],
        [0.9195, 0.2665, 0.3111],
        [0.0743, 0.2133, 0.4411]])

tensor([[0.1538, 0.0637, 0.3470],
        [0.5351, 0.1194, 0.0084],
        [0.0719, 0.0559, 0.3600]])



In [16]:
print(f"tensor.mul(tensor) \n {tensor.matmul(tensor.T)} \n")

# Alternative syntax:

print(f" t1 @ t2 \n {t1 @ t2} \n")

tensor.mul(tensor) 
 tensor([[2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.],
        [2., 2., 2., 2.]]) 

 t1 @ t2 
 tensor([[0.3681, 0.2329, 0.4952],
        [0.7166, 0.3099, 0.6546],
        [0.8045, 0.5509, 1.2778]]) 



In [19]:
print(tensor, "\n" )

tensor.add_(5)
print(tensor)


tensor([[1., 0., 0., 1.],
        [1., 0., 0., 1.],
        [1., 0., 0., 1.],
        [1., 0., 0., 1.]]) 

tensor([[6., 5., 5., 6.],
        [6., 5., 5., 6.],
        [6., 5., 5., 6.],
        [6., 5., 5., 6.]])


In [ ]:
t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()   # this method is from torch to numpy
print(f"t: {n}")



t: tensor([1., 1., 1., 1., 1.])
t: [1. 1. 1. 1. 1.]


In [24]:
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


思考一下，为什么t变了，n也会变？

ans：因为他们指向同一片内存。再思考一下，为什么指向同一片内存，他们的数据类型可以不一样？

In [25]:
n = np.ones(5)
t = torch.from_numpy # this method is from numpy to torch

In [27]:
np.add(n,1,out=n)   # there will be a difference without out=n, you can try it
print(f"n: {n}")
print(f"t: {t}")


n: [3. 3. 3. 3. 3.]
t: <built-in method from_numpy of type object at 0x12bec5128>
